In [20]:
import pandas as pd
import numpy as np

In [21]:
df=pd.read_csv(r"C:\Users\aayus\Desktop\RNN\DateFruit_Dataset.csv")

In [22]:
df.head()

,AREA,PERIMETER,MAJOR_AXIS,MINOR_AXIS,ECCENTRICITY,EQDIASQ,SOLIDITY,CONVEX_AREA,EXTENT,ASPECT_RATIO,...,KurtosisRR,KurtosisRG,KurtosisRB,EntropyRR,EntropyRG,EntropyRB,ALLdaub4RR,ALLdaub4RG,ALLdaub4RB,Class
0,422163,2378.908,837.8484,645.6693,0.6373,733.1539,0.9947,424428,0.7831,1.2976,...,3.2370,2.9574,4.2287,-59191263232,-50714214400,-39922372608,58.7255,54.9554,47.8400,BERHI
1,338136,2085.144,723.8198,595.2073,0.5690,656.1464,0.9974,339014,0.7795,1.2161,...,2.6228,2.6350,3.1704,-34233065472,-37462601728,-31477794816,50.0259,52.8168,47.8315,BERHI
2,526843,2647.394,940.7379,715.3638,0.6494,819.0222,0.9962,528876,0.7657,1.3150,...,3.7516,3.8611,4.7192,-93948354560,-74738221056,-60311207936,65.4772,59.2860,51.9378,BERHI
3,416063,2351.210,827.9804,645.2988,0.6266,727.8378,0.9948,418255,0.7759,1.2831,...,5.0401,8.6136,8.2618,-32074307584,-32060925952,-29575010304,43.3900,44.1259,41.1882,BERHI
4,347562,2160.354,763.9877,582.8359,0.6465,665.2291,0.9908,350797,0.7569,1.3108,...,2.7016,2.9761,4.4146,-39980974080,-35980042240,-25593278464,52.7743,50.9080,42.6666,BERHI


In [23]:
X= df.drop('Class', axis=1)
y= df['Class']

In [24]:
from sklearn.preprocessing import StandardScaler,LabelEncoder
le = LabelEncoder()
y = le.fit_transform(y)

In [25]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [26]:
scaler= StandardScaler()
X_train= scaler.fit_transform(X_train)
X_test= scaler.transform(X_test)


In [27]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

In [28]:
X_train_tesnor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)

X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

In [29]:
train_dataset = TensorDataset(X_train_tesnor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

In [30]:
train_loader= DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader= DataLoader(test_dataset, batch_size=32, shuffle=False)

In [31]:
class ANN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(ANN, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, output_size)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        out = self.softmax(out)
        return out

In [33]:
model = ANN(input_size=X_train.shape[1], hidden_size=64, output_size=len(np.unique(y)))
criterion= nn.CrossEntropyLoss()
optimizer= optim.Adam(model.parameters(), lr=0.001)
criterion= nn.CrossEntropyLoss()
optimizer= optim.Adam(model.parameters(), lr=0.001)


In [39]:
epochs = 100
for epoch in range(epochs):
    model.train()
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        correct = 0
        total = 0
        for inputs, labels in test_loader:
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}, Accuracy: {accuracy:.2f}%')

Epoch [1/100], Loss: 1.1666, Accuracy: 92.22%
Epoch [2/100], Loss: 1.1657, Accuracy: 92.22%
Epoch [3/100], Loss: 1.2183, Accuracy: 92.22%
Epoch [4/100], Loss: 1.1682, Accuracy: 91.67%
Epoch [5/100], Loss: 1.2381, Accuracy: 92.22%
Epoch [6/100], Loss: 1.1695, Accuracy: 92.22%
Epoch [7/100], Loss: 1.1657, Accuracy: 91.67%
Epoch [8/100], Loss: 1.2384, Accuracy: 92.22%
Epoch [9/100], Loss: 1.1663, Accuracy: 91.67%
Epoch [10/100], Loss: 1.1671, Accuracy: 91.67%
Epoch [11/100], Loss: 1.1675, Accuracy: 92.22%
Epoch [12/100], Loss: 1.1705, Accuracy: 92.22%
Epoch [13/100], Loss: 1.1754, Accuracy: 92.78%
Epoch [14/100], Loss: 1.1654, Accuracy: 92.22%
Epoch [15/100], Loss: 1.2286, Accuracy: 92.22%
Epoch [16/100], Loss: 1.1883, Accuracy: 92.78%
Epoch [17/100], Loss: 1.1691, Accuracy: 92.78%
Epoch [18/100], Loss: 1.1659, Accuracy: 92.78%
Epoch [19/100], Loss: 1.1665, Accuracy: 92.78%
Epoch [20/100], Loss: 1.1659, Accuracy: 93.33%
Epoch [21/100], Loss: 1.1685, Accuracy: 92.78%
Epoch [22/100], Loss: 

In [40]:
model.eval()

total = 0
correct = 0
with torch.no_grad():
    for inputs, labels in test_loader:
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
accuracy = 100 * correct / total
print(f'Accuracy: {accuracy:.2f}%')

Accuracy: 93.33%
